# Notebook 10: SHAP & LIME Analysis
## Car Price Prediction Project

**Objective:** Explain model predictions using SHAP (global + local) and LIME (local).

- **SHAP Global** → Which features matter most overall?
- **SHAP Local (Waterfall)** → Why did the model predict *this* price for *this* car?
- **LIME** → Local explanation for a single prediction

---

## 10.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install shap lime xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import lime
import lime.lime_tabular
import pickle
import warnings
warnings.filterwarnings('ignore')

# Initialize SHAP JS visualizations
shap.initjs()

## 10.2 Load Data and Best Model

In [ ]:
# Load data
X_train = pd.read_csv('/content/drive/MyDrive/X_train.csv')
X_test = pd.read_csv('/content/drive/MyDrive/X_test.csv')
y_train = pd.read_csv('/content/drive/MyDrive/y_train.csv').squeeze()
y_test = pd.read_csv('/content/drive/MyDrive/y_test.csv').squeeze()

# Load the XGBoost model (best manual model)
with open('/content/drive/MyDrive/xgboost_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

print(f"Data loaded: {X_train.shape[0]} train, {X_test.shape[0]} test")
print(f"Model: XGBoost")

---
## 10.3 SHAP — Global Explanation

**Global SHAP** shows which features are most important *across the entire dataset*. This answers: **What drives car prices in general?**

In [ ]:
# Create SHAP explainer for XGBoost
explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base price): ${explainer.expected_value:,.0f}")

In [ ]:
# SHAP Summary Plot (Bar) — Global feature importance
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance (Bar)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Summary Plot (Beeswarm) — Shows direction of impact
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Beeswarm Plot — Feature Impact Direction', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top features by mean absolute SHAP value
mean_shap = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean |SHAP|': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False)

print("Top 10 Features by Mean Absolute SHAP Value:")
print("(Higher = more influence on predictions)")
print()
mean_shap.head(10)

### Interpreting Global SHAP

The bar chart shows which features have the **most influence on predictions overall**:
- Features at the top = most important for the model's decisions
- The beeswarm plot also shows **direction**: red dots on the right mean high feature values push the prediction **up** (higher price)

---
## 10.4 SHAP — Local Explanation (Waterfall Plot)

**Local SHAP** explains a single prediction: **Why did the model predict this specific price for this specific car?**

In [ ]:
# Pick a specific car to explain (e.g., index 0 in the test set)
sample_idx = 0

actual_price = y_test.iloc[sample_idx]
predicted_price = xgb_model.predict(X_test.iloc[[sample_idx]])[0]

print(f"Car #{sample_idx} from test set:")
print(f"  Actual price:    ${actual_price:,.0f}")
print(f"  Predicted price: ${predicted_price:,.0f}")
print(f"  Error:           ${abs(actual_price - predicted_price):,.0f}")
print(f"\nFeature values for this car:")
print(X_test.iloc[sample_idx].to_string())

In [ ]:
# SHAP Waterfall plot for this specific car
shap_explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[sample_idx].values,
    feature_names=X_test.columns.tolist()
)

plt.figure(figsize=(12, 10))
shap.waterfall_plot(shap_explanation, show=False)
plt.title(f'SHAP Waterfall — Car #{sample_idx} (Predicted: ${predicted_price:,.0f})', fontsize=14)
plt.tight_layout()
plt.show()

### Interpreting the Waterfall Plot

- **Base value** (E[f(x)]): The average predicted price across all cars
- **Red bars**: Features that **push the prediction UP** (increase price)
- **Blue bars**: Features that **push the prediction DOWN** (decrease price)
- **Final value** (f(x)): The actual prediction for this car

The waterfall shows exactly how the model arrived at its prediction, starting from the base value and adding/subtracting the contribution of each feature.

In [ ]:
# SHAP Force plot for the same car
shap.force_plot(explainer.expected_value, shap_values[sample_idx],
                X_test.iloc[sample_idx], matplotlib=True)
plt.title(f'SHAP Force Plot — Car #{sample_idx}')
plt.tight_layout()
plt.show()

In [ ]:
# Explain a second car (e.g., the most expensive in test set)
expensive_idx = y_test.idxmax()
expensive_pos = list(y_test.index).index(expensive_idx)

actual_price2 = y_test.iloc[expensive_pos]
predicted_price2 = xgb_model.predict(X_test.iloc[[expensive_pos]])[0]

print(f"\nMost Expensive Car in Test Set (index {expensive_pos}):")
print(f"  Actual price:    ${actual_price2:,.0f}")
print(f"  Predicted price: ${predicted_price2:,.0f}")

shap_explanation2 = shap.Explanation(
    values=shap_values[expensive_pos],
    base_values=explainer.expected_value,
    data=X_test.iloc[expensive_pos].values,
    feature_names=X_test.columns.tolist()
)

plt.figure(figsize=(12, 10))
shap.waterfall_plot(shap_explanation2, show=False)
plt.title(f'SHAP Waterfall — Most Expensive Car (Predicted: ${predicted_price2:,.0f})', fontsize=14)
plt.tight_layout()
plt.show()

---
## 10.5 SHAP Dependence Plots

In [ ]:
# Top 4 features dependence plots
top4 = mean_shap.head(4)['Feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top4):
    shap.dependence_plot(feat, shap_values, X_test, ax=axes[i], show=False)
    axes[i].set_title(f'SHAP Dependence: {feat}')

plt.suptitle('SHAP Dependence Plots — Top 4 Features', fontsize=14)
plt.tight_layout()
plt.show()

---
## 10.6 LIME — Local Explanation

LIME explains individual predictions by creating a simple local model around the prediction.

In [ ]:
# Create LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    mode='regression',
    verbose=True
)

print("LIME explainer created.")

In [ ]:
# Explain the same car as SHAP (index 0)
lime_exp = lime_explainer.explain_instance(
    X_test.iloc[sample_idx].values,
    xgb_model.predict,
    num_features=15
)

print(f"LIME Explanation for Car #{sample_idx}")
print(f"Predicted price: ${predicted_price:,.0f}")
print(f"\nTop contributing features:")
for feature, weight in lime_exp.as_list():
    direction = '+' if weight > 0 else ''
    print(f"  {feature}: {direction}{weight:,.0f}")

In [ ]:
# LIME visualization
fig = lime_exp.as_pyplot_figure()
fig.set_size_inches(12, 8)
plt.title(f'LIME Explanation — Car #{sample_idx} (Predicted: ${predicted_price:,.0f})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# LIME HTML (interactive — works in Colab)
lime_exp.show_in_notebook(show_table=True)

In [ ]:
# LIME for the most expensive car
lime_exp2 = lime_explainer.explain_instance(
    X_test.iloc[expensive_pos].values,
    xgb_model.predict,
    num_features=15
)

print(f"LIME Explanation for Most Expensive Car")
print(f"Predicted price: ${predicted_price2:,.0f}")
print(f"\nTop contributing features:")
for feature, weight in lime_exp2.as_list():
    direction = '+' if weight > 0 else ''
    print(f"  {feature}: {direction}{weight:,.0f}")

fig = lime_exp2.as_pyplot_figure()
fig.set_size_inches(12, 8)
plt.title(f'LIME Explanation — Most Expensive Car (Predicted: ${predicted_price2:,.0f})', fontsize=14)
plt.tight_layout()
plt.show()

---
## 10.7 SHAP vs LIME Comparison

In [ ]:
# Compare top features from SHAP and LIME for the same car
print(f"Car #{sample_idx} — Predicted: ${predicted_price:,.0f}")
print(f"{'='*60}")

# SHAP top features
shap_contributions = pd.DataFrame({
    'Feature': X_test.columns,
    'SHAP Value': shap_values[sample_idx]
}).sort_values('SHAP Value', key=abs, ascending=False)

print(f"\nTop 10 SHAP contributions:")
for _, row in shap_contributions.head(10).iterrows():
    direction = '+' if row['SHAP Value'] > 0 else ''
    print(f"  {row['Feature']}: {direction}${row['SHAP Value']:,.0f}")

# LIME top features
print(f"\nTop 10 LIME contributions:")
for feature, weight in lime_exp.as_list()[:10]:
    direction = '+' if weight > 0 else ''
    print(f"  {feature}: {direction}${weight:,.0f}")

print(f"\nKey difference:")
print(f"  SHAP: Based on game theory, considers all feature interactions")
print(f"  LIME: Builds a local linear model around the prediction")
print(f"  Both aim to explain the same prediction but may rank features differently")

---
## 10.8 Summary for Presentation

### SHAP Interpretation

**Global (Slide 6 — Feature Importance):**
- The SHAP bar chart shows which features influence predictions the most
- Top features are the ones the model relies on most to predict car prices

**Local (Slide 6 — Waterfall Plot):**
- Each feature either **pushes the price up** (positive SHAP, red) or **pulls it down** (negative SHAP, blue)
- The waterfall starts from a base value (average prediction) and shows how each feature adjusts the final prediction

### LIME Interpretation
- LIME explains one prediction at a time by fitting a simple local model
- Green bars = features increasing predicted price
- Red bars = features decreasing predicted price

### How to Interpret for Presentation

| Method | Scope | What It Shows |
|--------|-------|---------------|
| **SHAP Global** | All predictions | Most important features overall |
| **SHAP Local** | One prediction | How features contribute to a specific car's price |
| **LIME** | One prediction | Similar to SHAP local but uses a different algorithm |

**Positive SHAP value** → increases predicted price  
**Negative SHAP value** → decreases predicted price  
**Larger magnitude** → stronger influence